# 01 — Phase 0: Dataset Preparation

**AI Media Authenticity / Deepfake Detection Platform**

This notebook is the entry point for the whole ML pipeline. It:

1. Detects whether we're running on Kaggle or locally
2. Inspects `/kaggle/input/` (if on Kaggle) to find the three mounted datasets
3. Locates each dataset's files on disk (no images/videos are opened yet — paths only)
4. Verifies the files actually exist and reports basic structure/class counts
5. Builds a **balanced 4,000-image subset** (2,000 real + 2,000 fake) for the image pipeline
6. Creates a **stratified 80/10/10 train/val/test split** and saves it to CSV
7. Runs the same discovery pass for the signature and video datasets, honestly reporting
   label reliability instead of assuming folder names mean what we hope they mean

Everything here operates on **filepaths and labels only** — no image, video, or signature
file is loaded into memory in this notebook. That happens lazily, batch-by-batch, in the
model-specific notebooks (02/03/04).

**Datasets used:**
| Modality | Dataset | Kaggle slug |
|---|---|---|
| Image | 140k Real and Fake Faces | `xhlulu/140k-real-and-fake-faces` |
| Signature | Real/Fake Signature Datasets | `emrahaydemr/realfake-signature-datasets` |
| Video | Deep Fake Detection (DFD) Entire Original Dataset | `sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset` |

> If you're running this on Kaggle, add all three datasets to the notebook via
> **"+ Add Input"** before running. If running locally, see `ml/README.md` for the
> Kaggle CLI commands to download small subsets into `ml/datasets/`.


## 1. Imports and project path bootstrap

In [1]:
!git clone https://github.com/mikasa16ren-lgtm/ml.git /kaggle/working/ml

Cloning into '/kaggle/working/ml'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 42 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 39.46 KiB | 734.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
import sys
sys.path.insert(0, "/kaggle/working/ml")

In [3]:
import os

print(os.path.exists("/kaggle/working/ml/src"))
print(os.listdir("/kaggle/working/ml/src"))

True
['xai', 'training', 'utils', 'models', '__pycache__', 'data', 'config.py', '__init__.py', 'evaluation']


In [4]:
%cd /kaggle/working/ml

/kaggle/working/ml


In [5]:
from src.utils.kaggle_utils import list_kaggle_inputs

for path in list_kaggle_inputs():
    print(path)

/kaggle/input/datasets
/kaggle/input/datasets/sanikatiwarekar
/kaggle/input/datasets/xhlulu
/kaggle/input/datasets/emrahaydemr
/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset
/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets
/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences
/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences
/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences
/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake
/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake
/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid
/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/re

In [6]:
from src.config import (
    IMAGE_DATASET_DIR,
    SIGNATURE_DATASET_DIR,
    VIDEO_DATASET_DIR
)

print("IMAGE DATASET :", IMAGE_DATASET_DIR)
print("SIGNATURE DATASET :", SIGNATURE_DATASET_DIR)
print("VIDEO DATASET :", VIDEO_DATASET_DIR)

IMAGE DATASET : /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
SIGNATURE DATASET : /kaggle/input/datasets/emrahaydemr/realfake-signature-datasets
VIDEO DATASET : /kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset


In [7]:
import sys
import os
from pathlib import Path

import pandas as pd

# --- Locate the ml/ project root regardless of where Jupyter was launched from ---
# We walk upward from the current working directory until we find a folder
# containing 'src/config.py'. This makes the notebook portable across:
#   - Kaggle notebooks (cwd = /kaggle/working)
#   - a local `jupyter lab` launched from ml/ or ml/notebooks/
def find_ml_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(6):
        if (current / "src" / "config.py").exists():
            return current
        current = current.parent
    raise FileNotFoundError(
        "Could not locate the 'ml/' project root (looked for src/config.py "
        "up to 6 parent directories above the current working directory). "
        "If running on Kaggle, add this notebook's 'ml' folder as a utility "
        "script/dataset, or copy ml/src into /kaggle/working/src."
    )

ML_ROOT = find_ml_root(Path.cwd())
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

print(f"ML project root resolved to: {ML_ROOT}")


ML project root resolved to: /kaggle/working/ml


In [8]:
from src.utils import set_seed, is_kaggle_env, list_kaggle_inputs
from src import config
from src.data import (
    build_image_metadata, ImageDatasetNotFoundError,
    build_signature_metadata, signature_label_reliability_report, SignatureDatasetNotFoundError,
    build_video_metadata, VideoDatasetNotFoundError,
    build_balanced_subset, stratified_split,
)

set_seed(config.RANDOM_STATE)
pd.set_option("display.max_colwidth", 80)
print("Imports OK. Random seed set to", config.RANDOM_STATE)


Imports OK. Random seed set to 42


## 2. Detect environment and inspect `/kaggle/input/`

In [9]:
print("Running on Kaggle:", is_kaggle_env())

kaggle_inputs = list_kaggle_inputs()
if kaggle_inputs:
    print(f"\nFound {len(kaggle_inputs)} dataset(s) mounted under /kaggle/input:")
    for p in kaggle_inputs:
        print(" -", p.name)
else:
    print("\nNot running on Kaggle (or no datasets mounted yet).")
    print("Falling back to local directories under:", config.DATASETS_ROOT)


Running on Kaggle: True

Found 22 dataset(s) mounted under /kaggle/input:
 - datasets
 - sanikatiwarekar
 - xhlulu
 - emrahaydemr
 - deep-fake-detection-dfd-entire-original-dataset
 - 140k-real-and-fake-faces
 - realfake-signature-datasets
 - DFD_original sequences
 - DFD_manipulated_sequences
 - DFD_manipulated_sequences
 - real_vs_fake
 - real-vs-fake
 - valid
 - test
 - train
 - fake
 - real
 - fake
 - real
 - fake
 - real
 - Signature Images


In [10]:
from pathlib import Path

paths = {
    "IMAGE": Path("/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces"),
    "VIDEO": Path("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset"),
    "SIGNATURE": Path("/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets"),
}

for name, path in paths.items():
    print(f"{name}: {path.exists()}")

    if path.exists():
        print("   Contents:")
        for item in list(path.iterdir())[:10]:
            print("    -", item.name)

IMAGE: True
   Contents:
    - valid.csv
    - real_vs_fake
    - train.csv
    - test.csv
VIDEO: True
   Contents:
    - DFD_original sequences
    - DFD_manipulated_sequences
SIGNATURE: True
   Contents:
    - Signature Images
    - Person Details.csv


## 3. Locate each dataset

`src/config.py` already ran auto-discovery at import time (matching folder-name
keywords against `/kaggle/input/*` first, then falling back to `ml/datasets/<name>/`
locally). We just print what was resolved.


In [11]:
print("Centralized dataset paths (from src/config.py):\n")
for name, value in config.summary().items():
    print(f"  {name:28s}: {value}")

print("\nRaw path objects:")
print("  IMAGE_DATASET_DIR     :", config.IMAGE_DATASET_DIR)
print("  IMAGE_DATA_DIR        :", config.IMAGE_DATA_DIR)
print("  IMAGE_TRAIN_CSV       :", config.IMAGE_TRAIN_CSV)
print("  IMAGE_VALID_CSV       :", config.IMAGE_VALID_CSV)
print("  IMAGE_TEST_CSV        :", config.IMAGE_TEST_CSV)
print("  SIGNATURE_DATASET_DIR :", config.SIGNATURE_DATASET_DIR)
print("  SIGNATURE_IMAGE_DIR   :", config.SIGNATURE_IMAGE_DIR)
print("  SIGNATURE_METADATA_CSV:", config.SIGNATURE_METADATA_CSV)
print("  VIDEO_DATASET_DIR     :", config.VIDEO_DATASET_DIR)


Centralized dataset paths (from src/config.py):

  is_kaggle                   : True
  image_dataset_dir           : /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
  signature_dataset_dir       : /kaggle/input/datasets/emrahaydemr/realfake-signature-datasets
  video_dataset_dir           : /kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset
  image_size                  : 224
  batch_size                  : 16
  num_epochs                  : 4
  vit_backbone                : vit_tiny_patch16_224

Raw path objects:
  IMAGE_DATASET_DIR     : /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
  IMAGE_DATA_DIR        : /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
  IMAGE_TRAIN_CSV       : /kaggle/working/ml/datasets/metadata/image_train_split.csv
  IMAGE_VALID_CSV       : /kaggle/working/ml/datasets/metadata/image_val_split.csv
  IMAGE_TEST_CSV        : /kaggle/working/ml/datasets/metadata/image_test_split.csv
  SIGNATURE_DATASET_DI

## 4. Image dataset — load metadata, verify files, show structure

We build metadata directly from the filesystem (walking for `real/` and `fake/`
folders) rather than trusting a specific CSV schema, since dataset mirrors can
differ slightly. This cell will raise a clear, actionable error if the dataset
isn't mounted/downloaded yet — it will NOT silently continue with fake data.


In [12]:
try:
    image_df = build_image_metadata(config.IMAGE_DATASET_DIR)
    print(f"Found {len(image_df)} image files.")
    print("\nClass distribution (full dataset, before subsetting):")
    print(image_df['label'].value_counts())
    print("\nSample rows:")
    display(image_df.sample(min(5, len(image_df)), random_state=config.RANDOM_STATE))
    IMAGE_DATASET_AVAILABLE = True
except ImageDatasetNotFoundError as e:
    print("IMAGE DATASET NOT AVAILABLE:\n")
    print(e)
    image_df = None
    IMAGE_DATASET_AVAILABLE = False


Found 140000 image files.

Class distribution (full dataset, before subsetting):
label
real    70000
fake    70000
Name: count, dtype: int64

Sample rows:


,filepath,label
40665,/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-...,real
48520,/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-...,real
138403,/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-...,fake
130079,/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-...,fake
50146,/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-...,real


## 5. Build the balanced image subset (2,000 real + 2,000 fake)

Per the project brief: **do not** train on the full 140k-image dataset. We sample
a fixed, reproducible subset (`random_state=42`) and only ever refer to file paths —
no image is opened or loaded into memory here.


In [13]:
if IMAGE_DATASET_AVAILABLE:
    image_subset = build_balanced_subset(
        image_df,
        label_col="label",
        n_per_class=config.IMAGE_SUBSET_PER_CLASS,
        random_state=config.RANDOM_STATE,
        classes=config.IMAGE_CLASSES,
    )
    print(f"Balanced subset size: {len(image_subset)} "
          f"(target: {config.IMAGE_SUBSET_PER_CLASS * len(config.IMAGE_CLASSES)})")
    print(image_subset['label'].value_counts())
else:
    image_subset = None
    print("Skipping subset creation - image dataset not available.")


Balanced subset size: 4000 (target: 4000)
label
real    2000
fake    2000
Name: count, dtype: int64


## 6. Stratified 80/10/10 train/val/test split

Expected sizes for a 4,000-image subset: **3,200 train / 400 val / 400 test.**


In [14]:
if image_subset is not None:
    train_df, val_df, test_df = stratified_split(
        image_subset,
        label_col="label",
        ratios=config.IMAGE_SPLIT_RATIOS,
        random_state=config.RANDOM_STATE,
    )

    print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
    print("\nTrain label balance:\n", train_df['label'].value_counts())
    print("\nVal label balance:\n", val_df['label'].value_counts())
    print("\nTest label balance:\n", test_df['label'].value_counts())
else:
    train_df = val_df = test_df = None


Train: 3200  |  Val: 400  |  Test: 400

Train label balance:
 label
real    1600
fake    1600
Name: count, dtype: int64

Val label balance:
 label
real    200
fake    200
Name: count, dtype: int64

Test label balance:
 label
fake    200
real    200
Name: count, dtype: int64


In [15]:
if train_df is not None:
    config.IMAGE_TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(config.IMAGE_TRAIN_CSV, index=False)
    val_df.to_csv(config.IMAGE_VALID_CSV, index=False)
    test_df.to_csv(config.IMAGE_TEST_CSV, index=False)
    print("Saved:")
    print(" -", config.IMAGE_TRAIN_CSV)
    print(" -", config.IMAGE_VALID_CSV)
    print(" -", config.IMAGE_TEST_CSV)
else:
    print("Nothing to save - image dataset was not available.")


Saved:
 - /kaggle/working/ml/datasets/metadata/image_train_split.csv
 - /kaggle/working/ml/datasets/metadata/image_val_split.csv
 - /kaggle/working/ml/datasets/metadata/image_test_split.csv


## 7. Signature dataset — discovery + honest label-reliability check

**We do not assume folder/column names like "Gender" or "Age" indicate
genuine/forged/AI-generated.** Instead, `build_signature_metadata` scans each
file's path (relative to the dataset root) for keyword evidence of the three
target classes, and `signature_label_reliability_report` tells us plainly how
much of the dataset got a confident label vs. `"unknown"`.

If reliability is low, **we report that here rather than inventing labels** —
Notebook 03 will only train a classifier on this data if it's reliable enough.


In [16]:
try:
    signature_df = build_signature_metadata(config.SIGNATURE_DATASET_DIR)
    print(f"Found {len(signature_df)} signature image files.")
    print("\nRaw class distribution (from path-keyword inference):")
    print(signature_df['label'].value_counts())

    reliability = signature_label_reliability_report(signature_df)
    print("\nLabel reliability report:")
    for k, v in reliability.items():
        print(f"  {k}: {v}")

    if not reliability["reliable_enough_for_training"]:
        print(
            "\nWARNING: more than 15% of signature files could not be confidently "
            "labeled from their path. Inspect 'datasets/metadata/signature_metadata.csv' "
            "manually before training a classifier on this data - see the "
            "'matched_keyword' column (empty = unknown) and the actual folder names "
            "in the dataset to determine the real labeling scheme."
        )

    display(signature_df.sample(min(5, len(signature_df)), random_state=config.RANDOM_STATE))
    SIGNATURE_DATASET_AVAILABLE = True
except SignatureDatasetNotFoundError as e:
    print("SIGNATURE DATASET NOT AVAILABLE:\n")
    print(e)
    signature_df = None
    SIGNATURE_DATASET_AVAILABLE = False


Found 2816 signature image files.

Raw class distribution (from path-keyword inference):
label
unknown    2816
Name: count, dtype: int64

Label reliability report:
  total_files: 2816
  counts_per_label: {'unknown': 2816}
  unknown_count: 2816
  unknown_fraction: 1.0
  reliable_enough_for_training: False



,filepath,label,matched_keyword
450,/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets/Signature Ima...,unknown,
1174,/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets/Signature Ima...,unknown,
1192,/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets/Signature Ima...,unknown,
2426,/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets/Signature Ima...,unknown,
1644,/kaggle/input/datasets/emrahaydemr/realfake-signature-datasets/Signature Ima...,unknown,


In [17]:
if SIGNATURE_DATASET_AVAILABLE:
    config.SIGNATURE_METADATA_CSV.parent.mkdir(parents=True, exist_ok=True)
    signature_df.to_csv(config.SIGNATURE_METADATA_CSV, index=False)
    print("Saved:", config.SIGNATURE_METADATA_CSV)
else:
    print("Nothing to save - signature dataset was not available.")


Saved: /kaggle/working/ml/datasets/metadata/signature_metadata.csv


## 8. Video dataset — discovery (paths only, no decoding)

We only walk the directory tree for video file paths and infer real/fake from
folder-name keywords - **no video is opened or decoded in this notebook.**
Frame sampling happens lazily in Notebook 04, on a small subset of files.


In [18]:
try:
    video_df = build_video_metadata(config.VIDEO_DATASET_DIR)
    print(f"Found {len(video_df)} video files.")
    print("\nClass distribution (full dataset, before subsetting):")
    print(video_df['label'].value_counts())
    display(video_df.sample(min(5, len(video_df)), random_state=config.RANDOM_STATE))
    VIDEO_DATASET_AVAILABLE = True
except VideoDatasetNotFoundError as e:
    print("VIDEO DATASET NOT AVAILABLE:\n")
    print(e)
    video_df = None
    VIDEO_DATASET_AVAILABLE = False


Found 3431 video files.

Class distribution (full dataset, before subsetting):
label
fake    3068
real     363
Name: count, dtype: int64


,filepath,label
1575,/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-origin...,fake
1949,/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-origin...,fake
3241,/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-origin...,fake
3143,/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-origin...,fake
1861,/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-origin...,fake


In [19]:
if VIDEO_DATASET_AVAILABLE:
    video_subset = build_balanced_subset(
        video_df,
        label_col="label",
        n_per_class=config.VIDEO_SUBSET_PER_CLASS,
        random_state=config.RANDOM_STATE,
        classes=config.VIDEO_CLASSES,
    )
    video_metadata_csv = config.METADATA_DIR / "video_subset_metadata.csv"
    video_subset.to_csv(video_metadata_csv, index=False)
    print(f"Balanced video subset: {len(video_subset)} files "
          f"(target: {config.VIDEO_SUBSET_PER_CLASS * len(config.VIDEO_CLASSES)})")
    print(video_subset['label'].value_counts())
    print("Saved:", video_metadata_csv)
else:
    video_subset = None
    print("Skipping video subset creation - video dataset not available.")


Balanced video subset: 100 files (target: 100)
label
fake    50
real    50
Name: count, dtype: int64
Saved: /kaggle/working/ml/datasets/metadata/video_subset_metadata.csv


## 9. Final Phase 0 summary

In [20]:
print("=" * 60)
print("PHASE 0 - DATASET PREPARATION SUMMARY")
print("=" * 60)

print(f"""
IMAGE
  Available     : {IMAGE_DATASET_AVAILABLE}
  Subset size   : {len(image_subset) if image_subset is not None else 'N/A'}
  Train/Val/Test: {(len(train_df), len(val_df), len(test_df)) if train_df is not None else 'N/A'}

SIGNATURE
  Available             : {SIGNATURE_DATASET_AVAILABLE}
  Total files           : {len(signature_df) if signature_df is not None else 'N/A'}
  Reliable for training : {reliability['reliable_enough_for_training'] if SIGNATURE_DATASET_AVAILABLE else 'N/A'}

VIDEO
  Available   : {VIDEO_DATASET_AVAILABLE}
  Subset size : {len(video_subset) if video_subset is not None else 'N/A'}
""")

print("Next notebook: 02_Image_Deepfake_Detection.ipynb")


PHASE 0 - DATASET PREPARATION SUMMARY

IMAGE
  Available     : True
  Subset size   : 4000
  Train/Val/Test: (3200, 400, 400)

SIGNATURE
  Available             : True
  Total files           : 2816
  Reliable for training : False

VIDEO
  Available   : True
  Subset size : 100

Next notebook: 02_Image_Deepfake_Detection.ipynb
